# 基于 Chat Completions API 实现外部函数调用

**2023年6月20日，OpenAI 官方在 Chat Completions API 原有的三种不同角色设定（System, Assistant, User）基础上，新增了 Function Calling 功能。**

Function Calling = 让模型返回结构化函数调用，而不是自然语言。
你告诉模型有哪些工具 → 模型决定调用哪个 → 模型返回 tool_calls → 你本地执行 → 把结果返回模型。

`functions` 是 Chat Completion API 中的可选参数，用于提供函数定义。其目的是使 GPT 模型能够生成符合所提供定义的函数参数。请注意，API不会实际执行任何函数调用。开发人员需要使用GPT 模型输出来执行函数调用。

如果提供了`functions`参数，默认情况下，GPT 模型将决定在何时适当地使用其中一个函数。

可以通过将`function_call`参数设置为`{"name": "<insert-function-name>"}`来强制 API 使用指定函数。

同时，也支持通过将`function_call`参数设置为`"none"`来强制API不使用任何函数。

如果使用了某个函数，则响应中的输出将包含`"finish_reason": "function_call"`，以及一个具有该函数名称和生成的函数参数的`function_call`对象。

## 概述

本 Notebook 介绍了如何将 Chat Completions API 与外部函数结合使用，以扩展 GPT 模型的功能。包含以下2个部分：
- 如何使用 `functions` 参数
- 如何使用 `function_call` 参数
- 使用 GPT 模型生成函数和参数
- 实际执行 GPT 模型生成的函数（以 SQL 查询为例）
- ### 注意：本示例直接构造 HTTP 请求访问 OpenAI API，因此无需使用 openai Python SDK。

### 注意：智谱 对应的是 tools / tool_choice / tool_calls

In [1]:
!pip install scipy tenacity tiktoken termcolor openai requests

Looking in indexes: https://mirrors.cloud.tencent.com/pypi/simple/


In [2]:
import json
import requests
import os
from tenacity import retry, wait_random_exponential, stop_after_attempt
from termcolor import colored

from dotenv import load_dotenv

# 自动加载 .env 文件里的所有 key
load_dotenv()
GPT_MODEL = "glm-4.5-air"

### 定义工具函数

首先，让我们定义一些用于调用聊天完成 API 的实用工具，并维护和跟踪对话状态。

In [3]:
# 使用了retry库，指定在请求失败时的重试策略。
# 这里设定的是指数等待（wait_random_exponential），时间间隔的最大值为40秒，并且最多重试3次（stop_after_attempt(3)）。
# 定义一个函数chat_completion_request，主要用于发送 聊天补全 请求到OpenAI服务器

#### 【新增】
# 智谱模型 Function Calling
# 参考： https://www.bigmodel.cn/dev/howuse/functioncall

# deepseek-chat 模型 Function Calling
# 注意：当前版本 deepseek-chat 模型 Function Calling 功能效果不稳定，会出现循环调用、空回复的情况。
# 参考： https://api-docs.deepseek.com/zh-cn/guides/function_calling

### 使用 智谱模型 Function Calling 和 deepseek-chat 模型 Function Calling 注意替换模型名称，调用地址，和API KEY
@retry(wait=wait_random_exponential(multiplier=1, max=40), stop=stop_after_attempt(3))
def chat_completion_request(messages, tools=None, tool_choice="auto", model=GPT_MODEL):

    # 设定请求的header信息，包括 API_KEY
    headers = {
        "Content-Type": "application/json",
        "Authorization": "Bearer " + os.getenv("ZHIPUAI_API_KEY"),
    }

    # 设定请求的JSON数据，包括GPT 模型名和要进行补全的消息
    json_data = {"model": model, "messages": messages}

    # 如果传入了tools，将其加入到json_data中
    if tools is not None:
        json_data.update({"tools": tools})

    # tool_choice，将其加入到json_data中
    if tool_choice is None:
        json_data["tool_choice"] = "auto"
    else:
        json_data["tool_choice"] = tool_choice

    # 尝试发送POST请求到OpenAI服务器的chat/completions接口
    try:
        response = requests.post(
            "https://open.bigmodel.cn/api/paas/v4/chat/completions",
            headers=headers,
            json=json_data,
        )
        # 返回服务器的响应
        return response

    # 如果发送请求或处理响应时出现异常，打印异常信息并返回
    except Exception as e:
        print("Unable to generate ChatCompletion response")
        print(f"Exception: {e}")
        return e



In [4]:
# 定义一个函数pretty_print_conversation，用于打印消息对话内容
def pretty_print_conversation(messages):

    # 为不同角色设置不同的颜色
    role_to_color = {
        "system": "red",
        "user": "green",
        "assistant": "blue",
        "tools": "magenta",
    }

    # 遍历消息列表
    for message in messages:

        # 如果消息的角色是"system"，则用红色打印“content”
        if message["role"] == "system":
            print(colored(f"system: {message['content']}\n", role_to_color[message["role"]]))

        # 如果消息的角色是"user"，则用绿色打印“content”
        elif message["role"] == "user":
            print(colored(f"user: {message['content']}\n", role_to_color[message["role"]]))

        # 如果消息的角色是"assistant"，并且消息中包含"tool_choice"，则用蓝色打印"tool_choice"
        elif message["role"] == "assistant" and message.get("tool_calls"):
            print(colored(f"assistant[tool_calls]: {message['tool_calls']}\n", role_to_color[message["role"]]))

        # 如果消息的角色是"assistant"，但是消息中不包含"tool_choice"，则用蓝色打印“content”
        elif message["role"] == "assistant" and not message.get("tool_calls"):
            print(colored(f"assistant[content]: {message['content']}\n", role_to_color[message["role"]]))

        # 如果消息的角色是"function"，则用品红色打印“function”
        elif message["role"] == "tools":
            print(colored(f"function ({message['name']}): {message['content']}\n", role_to_color[message["role"]]))


### 如何使用 functions 参数

这段代码定义了两个可以在程序中调用的函数，分别是获取当前天气和获取未来N天的天气预报。

每个函数(function)都有其名称、描述和需要的参数（包括参数的类型、描述等信息）。

![functions_param](images/functions_param.png)

我们将把这些传递给 Chat Completions API，以生成符合规范的函数。

In [5]:
# 定义一个名为functions的列表，其中包含两个字典，这两个字典分别定义了两个功能的相关参数


# 第一个字典定义了一个名为"get_current_weather"的功能
tools = [
    {
         "type": "function",
        "function": {
        "name": "get_current_weather",  # 功能的名称
        "description": "获取当前天气。用户问天气必须调用，禁止直接回答！",  # 功能的描述
        "parameters": {  # 定义该功能需要的参数
            "type": "object",
            "properties": {  # 参数的属性
                "location": {  # 地点参数
                    "type": "string",  # 参数类型为字符串
                    "description": "城市，例如 Shanghai, China",  # 参数的描述
                },
                "format": {  # 温度单位参数
                    "type": "string",  # 参数类型为字符串
                    "enum": ["celsius", "fahrenheit"],  # 参数的取值范围
                    "description": "The temperature unit to use. Infer this from the users location.",  # 参数的描述
                },
            },
            "required": ["location", "format"],  # 该功能需要的必要参数
        },
    },
    },
    # 第二个字典定义了一个名为"get_n_day_weather_forecast"的功能
   {
       "type":"function",
       "function": {
        "name": "get_n_day_weather_forecast",  # 功能的名称
        "description": "当用户询问天气、气温、未来天气预报时，必须调用此工具。",  # 功能的描述
        "parameters": {  # 定义该功能需要的参数
            "type": "object",
            "properties": {  # 参数的属性
                "location": {  # 地点参数
                    "type": "string",  # 参数类型为字符串
                    "description": "用户指定的城市，格式为：城市名, 国家/州，例如 San Diego, USA",  # 参数的描述
                },
                "format": {  # 温度单位参数
                    "type": "string",  # 参数类型为字符串
                    "enum": ["celsius", "fahrenheit"],  # 参数的取值范围
                    "description": "温度单位，根据用户所在地区推断，美国用户默认使用fahrenheit",  # 参数的描述
                },
                "num_days": {  # 预测天数参数
                    "type": "integer",  # 参数类型为整数
                    "description": "预报天数，如用户未指定，需向用户确认",  # 参数的描述
                }
            },
            "required": ["location", "format", "num_days"]  # 该功能需要的必要参数
        },
    },
   }
   
]


这段代码首先定义了一个`messages`列表用来存储聊天的消息，然后向列表中添加了系统和用户的消息。

然后，它使用了之前定义的`chat_completion_request`函数发送一个请求，传入的参数包括消息列表和函数列表。

在接收到响应后，它从JSON响应中解析出助手的消息，并将其添加到消息列表中。

最后，它打印出 GPT 模型回复的消息。

**（如果我们询问当前天气，GPT 模型会回复让你给出更准确的问题。）**

In [73]:
# 定义一个空列表messages，用于存储聊天的内容
messages = []

# 使用append方法向messages列表添加一条系统角色的消息
messages.append({
    "role": "system",  # 消息的角色是"system"
    "content": "你必须调用工具回答天气问题，绝对不允许直接编造答案！参数缺失时自动填充默认值，不要反问用户！location默认Shanghai,China，num_days默认1，format默认celsius。"  # 消息的内容
})

# 向messages列表添加一条用户角色的消息
messages.append({
    "role": "user",  # 消息的角色是"user"
    "content": "What's the weather like today"  # 用户询问今天的天气情况
})

# 使用定义的chat_completion_request函数发起一个请求，传入messages和functions作为参数
chat_response = chat_completion_request(
    messages, tools=tools,
   tool_choice="auto"
)

# 解析返回的JSON数据，获取助手的回复消息
assistant_message = chat_response.json()["choices"][0]["message"]

# 将助手的回复消息添加到messages列表中
messages.append(assistant_message)


pretty_print_conversation(messages)

system: 你必须调用工具回答天气问题，绝对不允许直接编造答案！参数缺失时自动填充默认值，不要反问用户！location默认Shanghai,China，num_days默认1，format默认celsius。

user: What's the weather like today

assistant[content]: 
I'd be happy to help you check today's weather! However, I need to know your location to get the current weather information. Could you please tell me which city you're interested in?



**(我们需要提供更详细的信息，以便于 GPT 模型为我们生成适当的函数和对应参数。)**
## 使用 GPT 模型生成函数和对应参数
下面这段代码先向messages列表中添加了用户的位置信息。

然后再次使用了chat_completion_request函数发起请求，只是这次传入的消息列表已经包括了用户的新消息。

在获取到响应后，它同样从JSON响应中解析出助手的消息，并将其添加到消息列表中。

最后，打印出助手的新的回复消息。

In [74]:
# 向messages列表添加一条用户角色的消息，用户告知他们在苏格兰的格拉斯哥
messages.append({
    "role": "user",  # 消息的角色是"user"
    "content": "在上海"  # 用户的消息内容
})

# 再次使用定义的chat_completion_request函数发起一个请求，传入更新后的messages和functions作为参数
chat_response = chat_completion_request(
    messages, tools=tools,
    tool_choice="auto"  # 必须写 auto！
)

# 解析返回的JSON数据，获取助手的新的回复消息
assistant_message = chat_response.json()["choices"][0]["message"]

# 将助手的新的回复消息添加到messages列表中
messages.append(assistant_message)

pretty_print_conversation(messages)

system: 你必须调用工具回答天气问题，绝对不允许直接编造答案！参数缺失时自动填充默认值，不要反问用户！location默认Shanghai,China，num_days默认1，format默认celsius。

user: What's the weather like today

assistant[content]: 
I'd be happy to help you check today's weather! However, I need to know your location to get the current weather information. Could you please tell me which city you're interested in?

user: 在上海

assistant[tool_calls]: [{'function': {'arguments': '{"location":"Shanghai, China","format":"celsius"}', 'name': 'get_current_weather'}, 'id': 'call_-7666799816323627559', 'index': 0, 'type': 'function'}]



这段代码的逻辑大体与上一段代码相同，区别在于这次用户的询问中涉及到未来若干天（x天）的天气预报。

在获取到回复后，它同样从JSON响应中解析出助手的消息，并将其添加到消息列表中。

然后打印出助手的回复消息。

**（通过不同的prompt方式，我们可以让它针对我们告诉它的其他功能。）**

In [76]:
# 初始化一个空的messages列表
messages = []

# 向messages列表添加一条系统角色的消息，要求不做关于函数参数值的假设，如果用户的请求模糊，应该寻求澄清
messages.append({
    "role": "system",  # 消息的角色是"system"
    "content": "你是一个会调用工具的智能助手。当用户询问天气、气温、天气预报相关问题时，必须优先调用工具，不要直接回答。如果工具参数不完整，优先推断合理值（例如用户问“今天”的天气，默认num_days=1；用户在中国，默认format=celsius），无法推断时再向用户确认。"
})

# 向messages列表添加一条用户角色的消息，用户询问在未来x天内苏格兰格拉斯哥的天气情况
messages.append({
    "role": "user",  # 消息的角色是"user"
    "content": "what is the weather going to be like in Shanghai, China over the next x days"
})

# 使用定义的chat_completion_request函数发起一个请求，传入messages和functions作为参数
chat_response = chat_completion_request(
    messages, tools=tools,
    tool_choice="auto"  # 必须写 auto！
)

# 解析返回的JSON数据，获取助手的回复消息
assistant_message = chat_response.json()["choices"][0]["message"]

# 将助手的回复消息添加到messages列表中
messages.append(assistant_message)

# 打印助手的回复消息
pretty_print_conversation(messages)

system: 你是一个会调用工具的智能助手。当用户询问天气、气温、天气预报相关问题时，必须优先调用工具，不要直接回答。如果工具参数不完整，优先推断合理值（例如用户问“今天”的天气，默认num_days=1；用户在中国，默认format=celsius），无法推断时再向用户确认。

user: what is the weather going to be like in Shanghai, China over the next x days

assistant[content]: 
I'd be happy to get the weather forecast for Shanghai, China! However, you mentioned "x days" - could you please specify how many days you'd like the forecast for? For example, 3 days, 5 days, or 7 days?



**(GPT 模型再次要求我们澄清，因为它还没有足够的信息。在这种情况下，它已经知道预测的位置，但需要知道需要多少天的预测。)**

这段代码的主要目标是将用户指定的天数（5天）添加到消息列表中，然后再次调用chat_completion_request函数发起一个请求。

返回的响应中包含了助手对用户的回复，即未来5天的天气预报。

这个预报是基于用户指定的地点（上海）和天数（5天）生成的。

在代码的最后，它解析出返回的JSON响应中的第一个选项，这就是助手的回复消息。

In [77]:
# 向messages列表添加一条用户角色的消息，用户指定接下来的天数为5天
messages.append({
    "role": "user",  # 消息的角色是"user"
    "content": "5 days"
})

# 使用定义的chat_completion_request函数发起一个请求，传入messages和functions作为参数
chat_response = chat_completion_request(
    messages, tools=tools,
    tool_choice="auto"  # 必须写 auto！
)

# 解析返回的JSON数据，获取第一个选项
assistant_message = chat_response.json()["choices"][0]["message"]

# 将助手的回复消息添加到messages列表中
messages.append(assistant_message)

# 打印助手的回复消息
pretty_print_conversation(messages)

system: 你是一个会调用工具的智能助手。当用户询问天气、气温、天气预报相关问题时，必须优先调用工具，不要直接回答。如果工具参数不完整，优先推断合理值（例如用户问“今天”的天气，默认num_days=1；用户在中国，默认format=celsius），无法推断时再向用户确认。

user: what is the weather going to be like in Shanghai, China over the next x days

assistant[content]: 
I'd be happy to get the weather forecast for Shanghai, China! However, you mentioned "x days" - could you please specify how many days you'd like the forecast for? For example, 3 days, 5 days, or 7 days?

user: 5 days

assistant[tool_calls]: [{'function': {'arguments': '{"location":"Shanghai, China","format":"celsius","num_days":5}', 'name': 'get_n_day_weather_forecast'}, 'id': 'call_-7666789955078717170', 'index': 0, 'type': 'function'}]



#### 强制使用指定函数
我们可以通过使用`function_call`参数来强制GPT 模型使用指定函数，例如`get_n_day_weather_forecast`。

通过这种方式，可以让 GPT 模型学习如何使用该函数。

In [78]:
# 在这个代码单元中，我们强制GPT 模型使用get_n_day_weather_forecast函数
messages = []  # 创建一个空的消息列表

# 添加系统角色的消息
messages.append({
    "role": "system",  # 角色为系统
    "content": "你是一个会调用工具的智能助手。当用户询问天气、气温、天气预报相关问题时，必须优先调用工具，不要直接回答。如果工具参数不完整，优先推断合理值（例如用户问“今天”的天气，默认num_days=1；用户在中国，默认format=celsius），无法推断时再向用户确认。"
})

# 添加用户角色的消息
messages.append({
    "role": "user",  # 角色为用户
    "content": "Give me a weather report for San Diego, USA."
})

# 使用定义的chat_completion_request函数发起一个请求，传入messages、functions以及特定的function_call作为参数
chat_response = chat_completion_request(
    messages, tools=tools, tool_choice={"type": "function", "function": {"name": "get_n_day_weather_forecast"}}
)

# 解析返回的JSON数据，获取第一个选项
assistant_message = chat_response.json()["choices"][0]["message"]

# 将助手的回复消息添加到messages列表中
messages.append(assistant_message)

# 打印助手的回复消息
pretty_print_conversation(messages)

system: 你是一个会调用工具的智能助手。当用户询问天气、气温、天气预报相关问题时，必须优先调用工具，不要直接回答。如果工具参数不完整，优先推断合理值（例如用户问“今天”的天气，默认num_days=1；用户在中国，默认format=celsius），无法推断时再向用户确认。

user: Give me a weather report for San Diego, USA.

assistant[tool_calls]: [{'function': {'arguments': '{"location":"San Diego, USA","format":"fahrenheit","num_days":1}', 'name': 'get_n_day_weather_forecast'}, 'id': 'call_-7666798270135398634', 'index': 0, 'type': 'function'}]



## 执行 GPT 模型生成的函数

接着，我们将演示如何执行输入为 GPT 模型生成的函数，并利用这一点来实现一个可以帮助我们回答关于数据库的问题的代理。

为了简单起见，我们将使用[Chinook样本数据库](https://www.sqlitetutorial.net/sqlite-sample-database/)。

![chinook_db](images/chinook_db.jpeg)

*注意：* 在生产环境中，SQL生成可能存在较高风险，因为GPT 模型在生成正确的SQL方面并不完全可靠。

### 定义一个执行SQL查询的函数

首先，让我们定义一些有用的实用函数来从SQLite数据库中提取数据。

In [6]:
import sqlite3

conn = sqlite3.connect("data/testSQL.db")
print("Opened database successfully")

Opened database successfully


首先定义三个函数`get_table_names`、`get_column_names`和`get_database_info`，用于从数据库连接对象中获取数据库的表名、表的列名以及整体数据库的信息。

In [7]:
def get_table_names(conn):
    """返回一个包含所有表名的列表"""
    table_names = []  # 创建一个空的表名列表
    # 执行SQL查询，获取数据库中所有表的名字
    tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table';")
    # 遍历查询结果，并将每个表名添加到列表中
    for table in tables.fetchall():
        table_names.append(table[0])
    return table_names  # 返回表名列表


def get_column_names(conn, table_name):
    """返回一个给定表的所有列名的列表"""
    column_names = []  # 创建一个空的列名列表
    # 执行SQL查询，获取表的所有列的信息
    columns = conn.execute(f"PRAGMA table_info('{table_name}');").fetchall()
    # 遍历查询结果，并将每个列名添加到列表中
    for col in columns:
        column_names.append(col[1])
    return column_names  # 返回列名列表


def get_database_info(conn):
    """返回一个字典列表，每个字典包含一个表的名字和列信息"""
    table_dicts = []  # 创建一个空的字典列表
    # 遍历数据库中的所有表
    for table_name in get_table_names(conn):
        columns_names = get_column_names(conn, table_name)  # 获取当前表的所有列名
        # 将表名和列名信息作为一个字典添加到列表中
        table_dicts.append({"table_name": table_name, "column_names": columns_names})
    return table_dicts  # 返回字典列表

将数据库信息转换为 Python 字典类型

In [8]:
# 获取数据库信息，并存储为字典列表
database_schema_dict = get_database_info(conn)

# 将数据库信息转换为字符串格式，方便后续使用
database_schema_string = "\n".join(
    [
        f"Table: {table['table_name']}\nColumns: {', '.join(table['column_names'])}"
        for table in database_schema_dict
    ]
)

然后，定义一个函数`ask_database`。

目标是让 GPT 模型帮我们构造一个完整的 SQL 查询。

In [9]:
# 定义一个功能列表，其中包含一个功能字典，该字典定义了一个名为"ask_database"的功能，用于回答用户关于音乐的问题
tools = [
    {
        "type":"function",
        "function":{
             "name": "ask_database",
        "description": "Use this function to answer user questions about music. Output should be a fully formed SQL query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": f"""
                            SQL query extracting info to answer the user's question.
                            SQL should be written using this database schema:
                            {database_schema_string}
                            The query should be returned in plain text, not in JSON.
                            """,
                }
            },
            "required": ["query"]
        }
    }
    }
]

### 执行 SQL 查询

首先，定义两个函数`ask_database`和`execute_function_call`
- 前者用于实际执行 SQL 查询并返回结果
- 后者用于根据消息中的功能调用信息来执行相应的功能并获取结果

然后，创建一个消息列表，并向其中添加了一个系统消息和一个用户消息。系统消息的内容是指示对话的目标，用户消息的内容是用户的问题。

接着，使用`chat_completion_request`函数发出聊天请求并获取响应，然后从响应中提取出助手的消息并添加到消息列表中。

如果助手的消息中包含功能调用，那么就使用`execute_function_call`函数执行这个功能调用并获取结果，然后将结果作为一个功能消息添加到消息列表中。

最后，使用`pretty_print_conversation`函数打印出整个对话。

In [10]:
def ask_database(conn, query):
    """使用 query 来查询 SQLite 数据库的函数。"""
    try:
        results = str(conn.execute(query).fetchall())  # 执行查询，并将结果转换为字符串
    except Exception as e:  # 如果查询失败，捕获异常并返回错误信息
        results = f"query failed with error: {e}"
    return results  # 返回查询结果


def execute_function_call(message):
    """执行工具调用，处理SQL查询"""
    # 1. 正确读取第一个工具调用
    tool_call = message["tool_calls"][0]
    function_name = tool_call["function"]["name"]
    arguments = json.loads(tool_call["function"]["arguments"])
    
    # 2. 判断函数名
    if function_name == "ask_database":
        query = arguments["query"]
        results = ask_database(conn, query)
    else:
        results = f"Error: function {function_name} does not exist"
    
    return results


In [13]:
# 创建一个空的消息列表
messages = []

# 向消息列表中添加一个系统角色的消息，内容是 "Answer user questions by generating SQL queries against the Chinook Music Database."
messages.append({"role": "system", "content": "Answer user questions by generating SQL queries against the Chinook Music Database."})

# 向消息列表中添加一个用户角色的消息，内容是 "Hi, who are the top 5 artists by number of tracks?"
messages.append({"role": "user", "content": "Hi, who are the top 5 artists by number of tracks?"})

# 使用 chat_completion_request 函数获取聊天响应
chat_response = chat_completion_request(messages, tools)

# 从聊天响应中获取助手的消息
assistant_message = chat_response.json()["choices"][0]["message"]

# 将助手的消息添加到消息列表中
messages.append(assistant_message)

# 如果助手的消息中有功能调用
if assistant_message.get("tool_calls"):
    # 使用 execute_function_call 函数执行功能调用，并获取结果
    results = execute_function_call(assistant_message)
    tool_call = assistant_message["tool_calls"][0]
    # 将功能的结果作为一个功能角色的消息添加到消息列表中
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call["id"],
        "name": tool_call["function"]["name"],
        "content": results
    })
    # 第二次调用：模型根据 SQL 结果生成自然语言回答
    final_response = chat_completion_request(messages, tools=tools)
    final_message = final_response.json()["choices"][0]["message"]
    messages.append(final_message)

# 使用 pretty_print_conversation 函数打印对话
pretty_print_conversation(messages)

system: Answer user questions by generating SQL queries against the Chinook Music Database.

user: Hi, who are the top 5 artists by number of tracks?

assistant[tool_calls]: [{'function': {'arguments': '{"query":"SELECT a.Name AS ArtistName, COUNT(t.TrackId) AS TrackCount\\nFROM artists a\\nJOIN albums al ON a.ArtistId = al.ArtistId\\nJOIN tracks t ON al.AlbumId = t.AlbumId\\nGROUP BY a.ArtistId, a.Name\\nORDER BY TrackCount DESC\\nLIMIT 5;"}', 'name': 'ask_database'}, 'id': 'call_-7666676293064192787', 'index': 0, 'type': 'function'}]

assistant[content]: 
Here are the top 5 artists by number of tracks:

1. **Iron Maiden** - 213 tracks
2. **U2** - 135 tracks  
3. **Led Zeppelin** - 114 tracks
4. **Metallica** - 112 tracks
5. **Deep Purple** - 92 tracks

Iron Maiden leads the pack with significantly more tracks than the other artists, followed by U2, Led Zeppelin, Metallica, and Deep Purple.



In [14]:
# 向消息列表中添加一个用户的问题，内容是 "What is the name of the album with the most tracks?"
messages.append({"role": "user", "content": "What is the name of the album with the most tracks?"})

# 使用 chat_completion_request 函数获取聊天响应
chat_response = chat_completion_request(messages, tools)

# 从聊天响应中获取助手的消息
assistant_message = chat_response.json()["choices"][0]["message"]

# 将助手的消息添加到消息列表中
messages.append(assistant_message)

# 如果助手的消息中有功能调用
if assistant_message.get("tool_calls"):
    # 使用 execute_function_call 函数执行功能调用，并获取结果
    results = execute_function_call(assistant_message)
    tool_call = assistant_message["tool_calls"][0]
    # 将功能的结果作为一个功能角色的消息添加到消息列表中
    messages.append({"role": "tool",
        "tool_call_id": tool_call["id"],
        "name": tool_call["function"]["name"],
        "content": results})
    # 第二次调用：模型根据 SQL 结果生成自然语言回答
    final_response = chat_completion_request(messages, tools=tools)
    final_message = final_response.json()["choices"][0]["message"]
    messages.append(final_message)
    
# 使用 pretty_print_conversation 函数打印对话
pretty_print_conversation(messages)

system: Answer user questions by generating SQL queries against the Chinook Music Database.

user: Hi, who are the top 5 artists by number of tracks?

assistant[tool_calls]: [{'function': {'arguments': '{"query":"SELECT a.Name AS ArtistName, COUNT(t.TrackId) AS TrackCount\\nFROM artists a\\nJOIN albums al ON a.ArtistId = al.ArtistId\\nJOIN tracks t ON al.AlbumId = t.AlbumId\\nGROUP BY a.ArtistId, a.Name\\nORDER BY TrackCount DESC\\nLIMIT 5;"}', 'name': 'ask_database'}, 'id': 'call_-7666676293064192787', 'index': 0, 'type': 'function'}]

assistant[content]: 
Here are the top 5 artists by number of tracks:

1. **Iron Maiden** - 213 tracks
2. **U2** - 135 tracks  
3. **Led Zeppelin** - 114 tracks
4. **Metallica** - 112 tracks
5. **Deep Purple** - 92 tracks

Iron Maiden leads the pack with significantly more tracks than the other artists, followed by U2, Led Zeppelin, Metallica, and Deep Purple.

user: What is the name of the album with the most tracks?

assistant[tool_calls]: [{'function'